# Dukascopy: millions of EUR/USD ticks

This notebook downloads Dukascopy's hourly BI5 tick files, decodes
bid/ask quotes, and sends every midpoint observation to one XY line.
The overview is M4-decimated to the viewport; wheel-zooming restores
detail from the canonical tick series.

The default is five calendar days beginning 2024-01-02. Change
`DUKASCOPY_START`, `DUKASCOPY_DAYS`, `DUKASCOPY_HOURS`, or
`DUKASCOPY_SYMBOL` to explore
a longer period. The price divisor below is correct for EUR/USD and
most five-decimal FX pairs.

**Source:** [Dukascopy Historical Data Export](https://www.dukascopy.com/swiss/english/marketwatch/historical/).

Install beside XY with `python -m pip install numpy requests xy`.


In [ ]:
import lzma
import os
import time as time_module
from datetime import UTC, date, datetime, time, timedelta
from pathlib import Path

import numpy as np
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import xy

DATA_DIR = Path(os.getenv("XY_REAL_WORLD_DATA", "data")) / "dukascopy"
DATA_DIR.mkdir(parents=True, exist_ok=True)

SYMBOL = os.getenv("DUKASCOPY_SYMBOL", "EURUSD").upper()
START = date.fromisoformat(os.getenv("DUKASCOPY_START", "2024-01-02"))
DAYS = int(os.getenv("DUKASCOPY_DAYS", "5"))
HOURS = int(os.getenv("DUKASCOPY_HOURS", "24"))
PRICE_DIVISOR = float(os.getenv("DUKASCOPY_PRICE_DIVISOR", "100000"))
REQUEST_DELAY = float(os.getenv("DUKASCOPY_REQUEST_DELAY", "0.2"))
if DAYS <= 0 or not 1 <= HOURS <= 24 or PRICE_DIVISOR <= 0 or REQUEST_DELAY < 0:
    raise ValueError("use positive days/divisor, 1-24 hours, and a non-negative delay")

SESSION = requests.Session()
SESSION.headers["User-Agent"] = "xy-real-world-notebook/1.0"
SESSION.mount(
    "https://",
    HTTPAdapter(
        max_retries=Retry(
            total=6,
            backoff_factor=1.0,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods={"GET"},
            respect_retry_after_header=True,
        )
    ),
)

TICK_DTYPE = np.dtype(
    [
        ("millisecond", ">u4"),
        ("ask", ">u4"),
        ("bid", ">u4"),
        ("ask_volume", ">f4"),
        ("bid_volume", ">f4"),
    ]
)

In [ ]:
def load_hour(day: date, hour: int) -> tuple[np.ndarray, np.ndarray]:
    relative = f"{SYMBOL}/{day.year}/{day.month - 1:02d}/{day.day:02d}/{hour:02d}h_ticks.bi5"
    url = f"https://datafeed.dukascopy.com/datafeed/{relative}"
    path = DATA_DIR / relative
    if not path.exists():
        response = SESSION.get(url, timeout=120)
        time_module.sleep(REQUEST_DELAY)
        if response.status_code == 404:
            return np.empty(0, dtype="datetime64[ms]"), np.empty(0)
        response.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(response.content)

    compressed = path.read_bytes()
    if not compressed:
        return np.empty(0, dtype="datetime64[ms]"), np.empty(0)
    ticks = np.frombuffer(lzma.decompress(compressed), dtype=TICK_DTYPE)
    hour_start = datetime.combine(
        day,
        time(hour=hour),
        tzinfo=UTC,
    )
    epoch_ms = int(hour_start.timestamp() * 1000)
    timestamps = (epoch_ms + ticks["millisecond"].astype(np.int64)).astype("datetime64[ms]")
    midpoint = (ticks["ask"].astype(np.float64) + ticks["bid"].astype(np.float64)) / (
        2 * PRICE_DIVISOR
    )
    return timestamps, midpoint


timestamp_parts = []
midpoint_parts = []
for day_offset in range(DAYS):
    day = START + timedelta(days=day_offset)
    for hour in range(HOURS):
        timestamps, midpoint = load_hour(day, hour)
        if midpoint.size:
            timestamp_parts.append(timestamps)
            midpoint_parts.append(midpoint)

if not midpoint_parts:
    raise RuntimeError("the selected interval returned no Dukascopy ticks")

timestamps = np.concatenate(timestamp_parts)
midpoint = np.concatenate(midpoint_parts)
print(f"{midpoint.size:,} {SYMBOL} ticks from {timestamps[0]} through {timestamps[-1]}")

In [ ]:
chart = xy.line_chart(
    xy.line(
        timestamps,
        midpoint,
        name=f"{SYMBOL[:3]}/{SYMBOL[3:]} midpoint",
        color="#5eead4",
        width=1.5,
    ),
    xy.x_axis(label="UTC"),
    xy.y_axis(label=f"{SYMBOL[:3]}/{SYMBOL[3:]}"),
    xy.legend(),
    xy.theme(
        background="#07111f",
        text_color="#e2e8f0",
        grid_color="#1e293b",
        axis_color="#94a3b8",
    ),
    title=f"Dukascopy tick history · {midpoint.size:,} quotes",
    width=1150,
    height=560,
)
payload = chart.figure().build_payload()[0]
print("render tier:", payload["traces"][0]["tier"])
chart